# N26 — Tire Agent

This notebook builds the **Tire Agent**, the second sub-agent in the F1 multi-agent system.
Its job is to answer one question per stint: **how much tyre life is left before the degradation cliff?**

The agent wraps two ML artefacts from N09/N10:
- **TireDegTCN** — causal TCN trained on 2023–2025 tyre stints, one per compound (C1–C5 fine-tuned, C6 global)
- **MC Dropout** × 50 forward passes for P10/P50/P90 uncertainty quantification

Pipeline:

    stint_state (compound, tyre_life, recent laps)
           │
           ▼
    predict_tire_deg_tool ──► TireDegTCN single pass ──► deg_rate (s/lap)
    estimate_laps_to_cliff_tool ──► MC Dropout × 50  ──► P10 / P50 / P90
           │
           ▼
      tire_react_agent (LangGraph ReAct)
           │
           ▼
      TireOutput(deg_rate, laps_to_cliff_p10/p50/p90, warning_level)

Output consumed by N31 Orchestrator before calling the Pit Strategy Agent (N28).

Models loaded from `data/models/tire_degradation/`:

| File | Contents |
|---|---|
| `routing_config.json` | compound → bundle filename + window size |
| `tiredeg_C{1-5}_ft.pt` | fine-tuned bundle: state dict + scaler + hparams |
| `tiredeg_modelA_v4.pt` | global fallback (C6 / unknown) |
| `mc_dropout_calibration.json` | per-compound uncertainty sigma (s) |


---

Imports and `repo_root` walker — no notebook-specific logic.

## Step 0 — Setup & model loading

Load the per-compound TireDegTCN bundles exported by N10. Each `.pt` file is
self-contained: state dict, fitted scaler, feature list, window size, and
architecture hyperparameters. The `load_bundle()` function reconstructs the
model from those artefacts without importing from the training notebook.


In [1]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_openai import ChatOpenAI            # noqa: E402
from langgraph.prebuilt import create_react_agent  # noqa: E402

Causal dilated conv block with LayerNorm + GELU — reproduced from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [2]:
# TireDegTCN is redefined here because src/strategy/inference/tire_predictor.py
# contains the older EnhancedTCN (N09 architecture, BatchNorm + attention),
# which has a different state dict layout than the N10 fine-tuned bundles.

class CausalConv1dBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=0)
        self.norm = nn.LayerNorm(out_ch)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.pad(x, (self.pad, 0))
        x = self.conv(x)
        return self.drop(F.gelu(self.norm(x.transpose(1, 2)).transpose(1, 2)))




Residual wrapper: two `CausalConv1dBlock`s with identity skip — from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [3]:
class TCNResidualBlock(nn.Module):
    def __init__(self, ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.net  = nn.Sequential(
            CausalConv1dBlock(ch, ch, kernel_size, dilation, dropout),
            CausalConv1dBlock(ch, ch, kernel_size, dilation, dropout),
        )

    def forward(self, x):
        return F.relu(self.net(x) + x)

Full TCN: linear input projection → residual blocks (dilation 2^i) → scalar output — from [N10](../strategy/tire_degradation/N10_tiredeg_compound_finetuning.ipynb).

In [4]:
class TireDegTCN(nn.Module):
    def __init__(self, n_features, d_model=64, n_layers=4, kernel_size=3, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(n_features, d_model)
        self.blocks      = nn.ModuleList([
            TCNResidualBlock(d_model, kernel_size, 2**i, dropout)
            for i in range(n_layers)
        ])
        self.output_head = nn.Linear(d_model, 1)

    def forward(self, x, mask=None):
        x = self.input_proj(x).transpose(1, 2)
        for block in self.blocks:
            x = block(x)
        return self.output_head(x.transpose(1, 2)[:, -1, :]).squeeze(-1)


`TireAgentConfig` dataclass: resolves model paths, loads routing and calibration JSONs, exposes `load_bundle` / `load_all_bundles`.

In [5]:
@dataclass
class TireAgentConfig:
    """Runtime config for the Tire Agent.

    Resolves all model paths relative to the repo root, loads the routing and
    calibration JSON files once on init, and exposes load_bundle / load_all_bundles
    so the rest of the notebook never needs to touch raw paths or json.load calls.
    """

    n_mc: int = 50
    model_name: str = "local-model"

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent
        self._model_dir = root / "data" / "models" / "tire_degradation"
        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        with open(self._model_dir / "routing_config.json") as f:
            self.routing_cfg: dict = json.load(f)
        with open(self._model_dir / "mc_dropout_calibration.json") as f:
            self.mc_calibration: dict = json.load(f)

        self.mc_sigma_fallback = float(
            np.mean([v["mean_sigma_s"] for v in self.mc_calibration.values()])
        )

    def load_bundle(self, compound_id: str) -> dict:
        """Load a compound bundle (.pt) and attach an instantiated TireDegTCN in eval mode."""
        cfg    = self.routing_cfg[compound_id]
        bundle = torch.load(self._model_dir / cfg["bundle"], map_location="cpu", weights_only=False)
        model  = TireDegTCN(bundle["n_features"], **bundle["model_hparams"])
        model.load_state_dict(bundle["state_dict"])
        model.eval()
        bundle["model"] = model
        return bundle

    def load_all_bundles(self) -> dict:
        """Load all compounds defined in routing_config and return as {compound_id: bundle}."""
        return {cid: self.load_bundle(cid) for cid in self.routing_cfg}


Instantiate config, load all six compound bundles from `data/models/tire_degradation/`, and connect the LM Studio LLM.

In [6]:
CFG     = TireAgentConfig()
BUNDLES = CFG.load_all_bundles()

llm = ChatOpenAI(
    model=CFG.model_name,
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0,
)

Verify all bundles loaded correctly — window sizes, feature count, scaler type, and MC calibration coverage.

In [7]:
print(f"Compounds loaded : {list(BUNDLES.keys())}")
for cid, b in BUNDLES.items():
    print(f"  {cid}  window={b['window']:>2}  features={b['n_features']}  scaler={type(b['scaler']).__name__}")
print(f"MC calibration   : {list(CFG.mc_calibration.keys())}")
print(f"MC sigma fallback: {CFG.mc_sigma_fallback:.4f} s  (C1, C3)")
print(f"N_MC             : {CFG.n_mc}")
print(f"EXPORT_DIR       : {CFG.export_dir}")


Compounds loaded : ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']
  C1  window=25  features=42  scaler=StandardScaler
  C2  window=31  features=42  scaler=StandardScaler
  C3  window=30  features=42  scaler=StandardScaler
  C4  window=26  features=42  scaler=StandardScaler
  C5  window=22  features=42  scaler=StandardScaler
  C6  window=28  features=42  scaler=StandardScaler
MC calibration   : ['C2', 'C4', 'C5', 'C6']
MC sigma fallback: 0.1729 s  (C1, C3)
N_MC             : 50
EXPORT_DIR       : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\agents


### Step 0 — Results

Six compound bundles loaded successfully (C1–C6). All are fine-tuned N10 variants
except C6 which falls back to the global `tiredeg_modelA_v4.pt`. Window sizes
range from 22 laps (C5) to 31 (C2), reflecting how many past laps each compound
variant needs to make a reliable prediction.

MC Dropout calibration is available for C2, C4, C5, C6. C1 and C3 use the
cross-compound mean sigma as fallback — uncertainty estimates for those compounds
will be slightly wider than the calibrated ones.
